In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
NVIDIA GeForce RTX 5080


In [2]:
import os
import re

output_dir = "./qwen_latexocr_mathwriting_sft"

checkpoints = []
if os.path.exists(output_dir):
    for name in os.listdir(output_dir):
        m = re.match(r"checkpoint-(\d+)", name)
        if m:
            checkpoints.append((int(m.group(1)), os.path.join(output_dir, name)))

latest_checkpoint = sorted(checkpoints)[-1][1] if checkpoints else None
print("Latest checkpoint:", latest_checkpoint)

if latest_checkpoint is not None:
    scaler_path = os.path.join(latest_checkpoint, "scaler.pt")
    if os.path.exists(scaler_path):
        os.remove(scaler_path)
        print("Removed scaler.pt from", latest_checkpoint)


Latest checkpoint: C:\Colab_Notebooks\qwen_latexocr_mathwriting_sft\checkpoint-5600


In [3]:
import torch
from dataclasses import dataclass
from typing import List, Dict, Any
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model
from utils import build_train_datasets, SYSTEM_PROMPT

MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"

@dataclass
class VLMDataCollator:
    processor: AutoProcessor

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        texts = []
        images = []
        labels_text = []

        for ex in features:
            image = ex["image"]
            label = ex["label"]

            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image},
                        {"type": "text", "text": SYSTEM_PROMPT},
                    ],
                }
            ]

            prompt = self.processor.apply_chat_template(
                messages,
                tokenize = False,
                add_generation_prompt = True
            )

            texts.append(prompt)
            images.append(image)
            labels_text.append(label)

        model_inputs = self.processor(
            text = texts,
            images = images,
            padding = True,
            return_tensors = "pt"
        )

        label_tokens = self.processor.tokenizer(
            labels_text,
            padding = True,
            return_tensors = "pt",
            add_special_tokens = False
        )["input_ids"]

        # для Trainer упростим схему:
        # обучаем на prompt + answer
        full_texts = [t + l for t, l in zip(texts, labels_text)]
        full_inputs = self.processor(
            text = full_texts,
            images = images,
            padding = True,
            return_tensors = "pt"
        )

        input_ids = full_inputs["input_ids"]
        attention_mask = full_inputs["attention_mask"]

        labels = input_ids.clone()

        # Маскируем prompt-часть, чтобы loss считался только по answer
        for i, (prompt, answer) in enumerate(zip(texts, labels_text)):
            prompt_ids = self.processor.tokenizer(prompt, add_special_tokens=False)["input_ids"]
            prompt_len = len(prompt_ids)
            labels[i, :prompt_len] = -100

        full_inputs["labels"] = labels
        return full_inputs


def main():
    train_latex_ocr, train_combined, test_latex_ocr = build_train_datasets(max_mathwriting_samples=20000)

    # Вариант A: train only on linxy/LaTeX_OCR
    train_dataset = train_combined

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_NAME,
        quantization_config = quant_config,
        device_map = "auto",
        trust_remote_code = True
    )

    lora_config = LoraConfig(
        r = 16,
        lora_alpha = 32,
        lora_dropout = 0.05,
        bias = "none",
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    collator = VLMDataCollator(processor)

    args = TrainingArguments(
        output_dir = output_dir,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        learning_rate = 2e-4,
        num_train_epochs = 2,
        fp16 = True,
        logging_steps = 10,
        save_strategy = "steps",
        save_steps = 200,
        save_total_limit = 2,
        remove_unused_columns = False,
        report_to = "none"
    )

    trainer = Trainer(
        model = model,
        args = args,
        train_dataset = train_dataset,
        data_collator = collator
    )

    if latest_checkpoint is not None:
        trainer.train(resume_from_checkpoint=latest_checkpoint)
    else:
        trainer.train()
    trainer.save_model("./qwen_latexocr_mathwriting_sft")
    processor.save_pretrained("./qwen_latexocr_mathwriting_sft")


if __name__ == "__main__":
    main()


README.md: 0.00B [00:00, ?B/s]

C:\Users\artem\anaconda3\envs\mathocr\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\artem\.cache\huggingface\hub\datasets--linxy--LaTeX_OCR. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


full/train-00000-of-00001.parquet:   0%|          | 0.00/383M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/76318 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

trainable params: 7,372,800 || all params: 3,761,995,776 || trainable%: 0.1960


Step,Training Loss
5610,1.573577
5620,1.018597
5630,0.938194
5640,0.883221
5650,0.950364
5660,0.856011
5670,1.277677
5680,1.194489
5690,0.949354
5700,0.956471


C:\Users\artem\anaconda3\envs\mathocr\lib\site-packages\peft\utils\other.py:1394: UserWarning: Unable to fetch remote file due to the following error [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1017) - silently ignoring the lookup for the file config.json in Qwen/Qwen2.5-VL-3B-Instruct.
  warnings.warn(
C:\Users\artem\anaconda3\envs\mathocr\lib\site-packages\peft\utils\save_and_load.py:295: UserWarning: Could not find a config file in Qwen/Qwen2.5-VL-3B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(
